# Yoruba VITS — Inference Notebook

In [1]:
import os

# Paths — adjust if running from a different working directory
RUN_DIR       = "outputs/yoruba/vits_yoruba-April-29-2026_11+52PM-fd8dd03"
checkpoint_path = os.path.join(RUN_DIR, "checkpoint_250000.pth")
config_path     = os.path.join(RUN_DIR, "config.json")
speakers_path   = os.path.join(RUN_DIR, "speakers.pth")

print("checkpoint :", checkpoint_path)
print("config     :", config_path)
print("speakers   :", speakers_path)

checkpoint : outputs/yoruba/vits_yoruba-April-29-2026_11+52PM-fd8dd03/checkpoint_250000.pth
config     : outputs/yoruba/vits_yoruba-April-29-2026_11+52PM-fd8dd03/config.json
speakers   : outputs/yoruba/vits_yoruba-April-29-2026_11+52PM-fd8dd03/speakers.pth


In [2]:
import torch

# ── Monkey-patch: guard rational_quadratic_spline against empty-tensor inputs ──
# Identical to the patch applied in train_vits.py so inference uses the same
# corrected flow path.
import TTS.tts.layers.vits.transforms as _vits_transforms

_orig_rqs = _vits_transforms.rational_quadratic_spline

def _patched_rqs(inputs, *args, **kwargs):
    if inputs.numel() == 0:
        return inputs, torch.zeros_like(inputs)
    return _orig_rqs(inputs, *args, **kwargs)

_vits_transforms.rational_quadratic_spline = _patched_rqs

from TTS.utils.synthesizer import Synthesizer

print("TTS imports OK")


TTS imports OK


In [3]:
from TTS.tts.utils.speakers import SpeakerManager

use_cuda = torch.cuda.is_available()
print(f"CUDA available: {use_cuda}")

synthesizer = Synthesizer(
    tts_checkpoint=checkpoint_path,
    tts_config_path=config_path,
    tts_speakers_file=speakers_path,
    use_cuda=use_cuda,
)

# Coqui's Synthesizer stores tts_speakers_file but never injects it into the
# config before building the model, so SpeakerManager.init_from_config returns
# None.  Manually restore the speaker manager from the saved .pth file.
if synthesizer.tts_model.speaker_manager is None:
    synthesizer.tts_model.speaker_manager = SpeakerManager(
        speaker_id_file_path=speakers_path
    )

# Show available speaker names
sm = synthesizer.tts_model.speaker_manager
print(f"\nSpeakers ({sm.num_speakers}):")
for name in sorted(sm.speaker_names):
    print(" ", name)


CUDA available: True
 > Using model: vits
 > Setting up Audio Processor...
 | > sample_rate:22050
 | > resample:False
 | > num_mels:80
 | > log_func:np.log10
 | > min_level_db:0
 | > frame_shift_ms:None
 | > frame_length_ms:None
 | > ref_level_db:None
 | > fft_size:1024
 | > power:None
 | > preemphasis:0.0
 | > griffin_lim_iters:None
 | > signal_norm:None
 | > symmetric_norm:None
 | > mel_fmin:0
 | > mel_fmax:None
 | > pitch_fmin:None
 | > pitch_fmax:None
 | > spec_gain:20.0
 | > stft_pad_mode:reflect
 | > max_norm:1.0
 | > clip_norm:True
 | > do_trim_silence:False
 | > trim_db:60
 | > do_sound_norm:False
 | > do_amp_to_db_linear:True
 | > do_amp_to_db_mel:True
 | > do_rms_norm:False
 | > db_level:None
 | > stats_path:None
 | > base:10
 | > hop_length:256
 | > win_length:1024



Speakers (1):
  SPEAKER_00_Yoruba


In [4]:
import numpy as np
from IPython.display import display, Audio, HTML

def synthesize(text: str, speaker_name: str | None = None) -> np.ndarray:
    """Run synthesis and return a float32 waveform array."""
    wav_data = synthesizer.tts(text, speaker_name=speaker_name)
    return np.array(wav_data, dtype=np.float32)


def play(text: str, speaker_name: str | None = None) -> None:
    """Synthesize *text*, print it, and embed a playable audio widget."""
    waveform = synthesize(text, speaker_name=speaker_name)
    sr = synthesizer.output_sample_rate

    display(HTML(f"<b>Speaker:</b> {speaker_name or '(default)'}"))
    display(HTML(f"<blockquote>{text}</blockquote>"))
    display(Audio(waveform, rate=sr, normalize=True))
    print()


In [5]:
# ── Pick a default speaker ──────────────────────────────────────────────────
# Set to None to use the model default, or replace with any name from the list
# printed in the cell above.
DEFAULT_SPEAKER = sorted(synthesizer.tts_model.speaker_manager.speaker_names)[0]
print(f"Using speaker: {DEFAULT_SPEAKER}")

# ── Sample Yoruba sentences ──────────────────────────────────────────────────
sample_texts = [
    "Bí ó ṣe ń bá mi sọ̀rọ̀, mo ti sùn lọ fọnfọn, bí mo ṣe da ojú bolẹ̀. Nígbà náà ni ó fi ọwọ́ kàn mí, ó sì gbé mi dúró lórí ẹsẹ̀ mi.",
]

for text in sample_texts:
    play(text, speaker_name=DEFAULT_SPEAKER)

Using speaker: SPEAKER_00_Yoruba
 > Text splitted to sentences.
['Bí ó ṣe ń bá mi sọ̀rọ̀, mo ti sùn lọ fọnfọn, bí mo ṣe da ojú bolẹ̀.', 'Nígbà náà ni ó fi ọwọ́ kàn mí, ó sì gbé mi dúró lórí ẹsẹ̀ mi.']
 > Processing time: 1.150484323501587
 > Real-time factor: 0.10574304443948408


# Vietnamese VITS — Inference Notebook

In [13]:
import os

# Paths — adjust if running from a different working directory
RUN_DIR       = "outputs/vietnamese/vits_vietnamese-April-30-2026_09+08AM-2541a19"
checkpoint_path = os.path.join(RUN_DIR, "checkpoint_250000.pth")
config_path     = os.path.join(RUN_DIR, "config.json")
speakers_path   = os.path.join(RUN_DIR, "speakers.pth")

print("checkpoint :", checkpoint_path)
print("config     :", config_path)
print("speakers   :", speakers_path)

checkpoint : outputs/vietnamese/vits_vietnamese-April-30-2026_09+08AM-2541a19/checkpoint_250000.pth
config     : outputs/vietnamese/vits_vietnamese-April-30-2026_09+08AM-2541a19/config.json
speakers   : outputs/vietnamese/vits_vietnamese-April-30-2026_09+08AM-2541a19/speakers.pth


In [2]:
import torch

# ── Monkey-patch: guard rational_quadratic_spline against empty-tensor inputs ──
# Identical to the patch applied in train_vits.py so inference uses the same
# corrected flow path.
import TTS.tts.layers.vits.transforms as _vits_transforms

_orig_rqs = _vits_transforms.rational_quadratic_spline

def _patched_rqs(inputs, *args, **kwargs):
    if inputs.numel() == 0:
        return inputs, torch.zeros_like(inputs)
    return _orig_rqs(inputs, *args, **kwargs)

_vits_transforms.rational_quadratic_spline = _patched_rqs

from TTS.utils.synthesizer import Synthesizer

print("TTS imports OK")


TTS imports OK


In [14]:
from TTS.tts.utils.speakers import SpeakerManager

use_cuda = torch.cuda.is_available()
print(f"CUDA available: {use_cuda}")

synthesizer = Synthesizer(
    tts_checkpoint=checkpoint_path,
    tts_config_path=config_path,
    tts_speakers_file=speakers_path,
    use_cuda=use_cuda,
)

# Coqui's Synthesizer stores tts_speakers_file but never injects it into the
# config before building the model, so SpeakerManager.init_from_config returns
# None.  Manually restore the speaker manager from the saved .pth file.
if synthesizer.tts_model.speaker_manager is None:
    synthesizer.tts_model.speaker_manager = SpeakerManager(
        speaker_id_file_path=speakers_path
    )

# Show available speaker names
sm = synthesizer.tts_model.speaker_manager
print(f"\nSpeakers ({sm.num_speakers}):")
for name in sorted(sm.speaker_names):
    print(" ", name)


CUDA available: True
 > Using model: vits
 > Setting up Audio Processor...
 | > sample_rate:22050
 | > resample:False
 | > num_mels:80
 | > log_func:np.log10
 | > min_level_db:0
 | > frame_shift_ms:None
 | > frame_length_ms:None
 | > ref_level_db:None
 | > fft_size:1024
 | > power:None
 | > preemphasis:0.0
 | > griffin_lim_iters:None
 | > signal_norm:None
 | > symmetric_norm:None
 | > mel_fmin:0
 | > mel_fmax:None
 | > pitch_fmin:None
 | > pitch_fmax:None
 | > spec_gain:20.0
 | > stft_pad_mode:reflect
 | > max_norm:1.0
 | > clip_norm:True
 | > do_trim_silence:False
 | > trim_db:60
 | > do_sound_norm:False
 | > do_amp_to_db_linear:True
 | > do_amp_to_db_mel:True
 | > do_rms_norm:False
 | > db_level:None
 | > stats_path:None
 | > base:10
 | > hop_length:256
 | > win_length:1024



Speakers (1):
  default


In [15]:
import numpy as np
from IPython.display import display, Audio, HTML

def synthesize(text: str, speaker_name: str | None = None) -> np.ndarray:
    """Run synthesis and return a float32 waveform array."""
    wav_data = synthesizer.tts(text, speaker_name=speaker_name)
    return np.array(wav_data, dtype=np.float32)


def play(text: str, speaker_name: str | None = None) -> None:
    """Synthesize *text*, print it, and embed a playable audio widget."""
    waveform = synthesize(text, speaker_name=speaker_name)
    sr = synthesizer.output_sample_rate

    display(HTML(f"<b>Speaker:</b> {speaker_name or '(default)'}"))
    display(HTML(f"<blockquote>{text}</blockquote>"))
    display(Audio(waveform, rate=sr, normalize=True))
    print()


In [16]:
# ── Pick a default speaker ──────────────────────────────────────────────────
# Set to None to use the model default, or replace with any name from the list
# printed in the cell above.
DEFAULT_SPEAKER = sorted(synthesizer.tts_model.speaker_manager.speaker_names)[0]
print(f"Using speaker: {DEFAULT_SPEAKER}")

# ── Sample Vietnamese sentences ──────────────────────────────────────────────
sample_texts = [
    "Trên đường lên Giê-ru-sa-lem, Chúa Giê-xu đến ranh giới xứ Ga-li-lê và xứ Sa-ma-ri.",
]

for text in sample_texts:
    play(text, speaker_name=DEFAULT_SPEAKER)

Using speaker: default
 > Text splitted to sentences.
['Trên đường lên Giê-ru-sa-lem, Chúa Giê-xu đến ranh giới xứ Ga-li-lê và xứ Sa-ma-ri.']
 > Processing time: 0.09407925605773926
 > Real-time factor: 0.013945678687164882


# Hindi VITS — Inference Notebook

In [8]:
import os

# Paths — adjust if running from a different working directory
RUN_DIR       = "outputs/hindi/vits_hindi-May-02-2026_04+32PM-2541a19"
checkpoint_path = os.path.join(RUN_DIR, "checkpoint_235000.pth")
config_path     = os.path.join(RUN_DIR, "config.json")
speakers_path   = os.path.join(RUN_DIR, "speakers.pth")

print("checkpoint :", checkpoint_path)
print("config     :", config_path)
print("speakers   :", speakers_path)

checkpoint : outputs/hindi/vits_hindi-May-02-2026_04+32PM-2541a19/checkpoint_235000.pth
config     : outputs/hindi/vits_hindi-May-02-2026_04+32PM-2541a19/config.json
speakers   : outputs/hindi/vits_hindi-May-02-2026_04+32PM-2541a19/speakers.pth


In [2]:
import torch

# ── Monkey-patch: guard rational_quadratic_spline against empty-tensor inputs ──
# Identical to the patch applied in train_vits.py so inference uses the same
# corrected flow path.
import TTS.tts.layers.vits.transforms as _vits_transforms

_orig_rqs = _vits_transforms.rational_quadratic_spline

def _patched_rqs(inputs, *args, **kwargs):
    if inputs.numel() == 0:
        return inputs, torch.zeros_like(inputs)
    return _orig_rqs(inputs, *args, **kwargs)

_vits_transforms.rational_quadratic_spline = _patched_rqs

from TTS.utils.synthesizer import Synthesizer

print("TTS imports OK")


TTS imports OK


In [9]:
from TTS.tts.utils.speakers import SpeakerManager

use_cuda = torch.cuda.is_available()
print(f"CUDA available: {use_cuda}")

synthesizer = Synthesizer(
    tts_checkpoint=checkpoint_path,
    tts_config_path=config_path,
    tts_speakers_file=speakers_path,
    use_cuda=use_cuda,
)

# Coqui's Synthesizer stores tts_speakers_file but never injects it into the
# config before building the model, so SpeakerManager.init_from_config returns
# None.  Manually restore the speaker manager from the saved .pth file.
if synthesizer.tts_model.speaker_manager is None:
    synthesizer.tts_model.speaker_manager = SpeakerManager(
        speaker_id_file_path=speakers_path
    )

# Show available speaker names
sm = synthesizer.tts_model.speaker_manager
print(f"\nSpeakers ({sm.num_speakers}):")
for name in sorted(sm.speaker_names):
    print(" ", name)


CUDA available: True
 > Using model: vits
 > Setting up Audio Processor...
 | > sample_rate:22050
 | > resample:False
 | > num_mels:80
 | > log_func:np.log10
 | > min_level_db:0
 | > frame_shift_ms:None
 | > frame_length_ms:None
 | > ref_level_db:None
 | > fft_size:1024
 | > power:None
 | > preemphasis:0.0
 | > griffin_lim_iters:None
 | > signal_norm:None
 | > symmetric_norm:None
 | > mel_fmin:0
 | > mel_fmax:None
 | > pitch_fmin:None
 | > pitch_fmax:None
 | > spec_gain:20.0
 | > stft_pad_mode:reflect
 | > max_norm:1.0
 | > clip_norm:True
 | > do_trim_silence:False
 | > trim_db:60
 | > do_sound_norm:False
 | > do_amp_to_db_linear:True
 | > do_amp_to_db_mel:True
 | > do_rms_norm:False
 | > db_level:None
 | > stats_path:None
 | > base:10
 | > hop_length:256
 | > win_length:1024

Speakers (1):
  SPEAKER_00_Hindi


In [10]:
import numpy as np
from IPython.display import display, Audio, HTML

def synthesize(text: str, speaker_name: str | None = None) -> np.ndarray:
    """Run synthesis and return a float32 waveform array."""
    wav_data = synthesizer.tts(text, speaker_name=speaker_name)
    return np.array(wav_data, dtype=np.float32)


def play(text: str, speaker_name: str | None = None) -> None:
    """Synthesize *text*, print it, and embed a playable audio widget."""
    waveform = synthesize(text, speaker_name=speaker_name)
    sr = synthesizer.output_sample_rate

    display(HTML(f"<b>Speaker:</b> {speaker_name or '(default)'}"))
    display(HTML(f"<blockquote>{text}</blockquote>"))
    display(Audio(waveform, rate=sr, normalize=True))
    print()


In [11]:
# ── Pick a default speaker ──────────────────────────────────────────────────
# Set to None to use the model default, or replace with any name from the list
# printed in the cell above.
DEFAULT_SPEAKER = sorted(synthesizer.tts_model.speaker_manager.speaker_names)[0]
print(f"Using speaker: {DEFAULT_SPEAKER}")

# ── Sample Hindi sentences ──────────────────────────────────────────────────
sample_texts = [
    "जब दावीद ने नाबाल की मृत्यु का समाचार सुना"
]

for text in sample_texts:
    play(text, speaker_name=DEFAULT_SPEAKER)

Using speaker: SPEAKER_00_Hindi
 > Text splitted to sentences.
['जब दावीद ने नाबाल की मृत्यु का समाचार सुना']
 > Processing time: 0.09700298309326172
 > Real-time factor: 0.02393594200096711
